# TFNBS vs classical NBS — EEG demonstration

Paired comparison of two EEG conditions (**fo** vs **fz**) using:

1. **TFNBS** (threshold-free) — integrates cluster statistics across a range of thresholds, so no manual threshold is required.
2. **Classical NBS** with fixed thresholds (2.1 and 2.75) — shows how results depend on the chosen threshold.

Data: 177 subjects × 19 electrodes × 7 frequency bands, reshaped to (n_subjects, 133, 133) block-diagonal connectivity matrices. Loaded from `datasets/eeg_dataframe_nansfilled.csv`.


In [ ]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

import numpy as np
import matplotlib.pyplot as plt

from conninfpy import eeg_utils, nbs_bct, compute_t_stat_diff, apply_tfnbs

%load_ext autoreload
%autoreload 2


### Loading EEG data for comparision

In [ ]:
path = "../../datasets/eeg_dataframe_nansfilled.csv"

stable_fo = eeg_utils.read_from_eeg_dataframe(path, cond_prefix='fo')
stable_fz = eeg_utils.read_from_eeg_dataframe(path, cond_prefix='fz')

data_fo = eeg_utils.reshape_eeg_data(stable_fo.data, reshape_bands=True) # Reshaping Data into M*N*N format
data_fz = eeg_utils.reshape_eeg_data(stable_fz.data, reshape_bands=True)


### Comparing between groups 

In [ ]:
diff_data = data_fo - data_fz
e = 0.4; h = 2

In [ ]:
%%time
# Two-step: compute paired t-stat from diffs, then apply TFNBS enhancement.
# Replaces the old combined compute_t_stat_tfnos_diffs convenience wrapper.
t_stat_dict = compute_t_stat_diff(diff_data)
tf_scores = apply_tfnbs(t_stat_dict, e=e, h=h, n=10)


In [ ]:
def plot_2_matrix(mat1, mat2, title1, title2, cmap = 'viridis'):
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6)) 
    
    axes = axes.flatten()
    axes[0].imshow(mat1, cmap=cmap)  
    axes[0].set_title(title1); axes[0].axis('off')  
    
    im2 = axes[1].imshow(mat2, cmap=cmap)
    axes[1].set_title(title2); axes[1].axis('off')
    
    fig.tight_layout()
    plt.show()

def plot_4_matrix(mat1, mat2, title1, title2, 
                  mat3, mat4, title3, title4, 
                  cmap = 'viridis'):
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14)) 
    
    axes = axes.flatten()
    axes[0].imshow(mat1, cmap=cmap)  
    axes[0].set_title(title1); axes[0].axis('off')  
    
    im2 = axes[1].imshow(mat2, cmap=cmap)
    axes[1].set_title(title2); axes[1].axis('off')

    im3 = axes[2].imshow(mat3, cmap=cmap)
    axes[2].set_title(title3); axes[2].axis('off')

    im4 = axes[3].imshow(mat4, cmap=cmap)
    axes[3].set_title(title4); axes[3].axis('off')
    
    fig.tight_layout()
    plt.show()

In [ ]:
plot_2_matrix(tf_scores['g2>g1'], tf_scores['g1>g2'],
              f"TFNBS score g2>g1 (e={e}, h={h})", f"TFNBS score g1>g2 (e={e}, h={h})")


### Computing NBS approach using thresholds = [2.1, 2.75]

In [ ]:
%%time
p_vals, _, _ = nbs_bct(data_fo, data_fz, threshold=2.1, test_type='paired', random_state=0)


In [ ]:
plot_2_matrix(-p_vals['g2>g1'], -p_vals['g1>g2'],
              f"NBS (threshold = 2.1) g2>g1", f"NBS (threshold = 2.1) g1>g2")

In [ ]:
%%time
p_vals, _, _ = nbs_bct(data_fo, data_fz, threshold=2.75, test_type='paired', random_state=0)


In [ ]:
plot_2_matrix(-p_vals['g2>g1'], -p_vals['g1>g2'],
              f"NBS (threshold = 2.75) g2>g1", f"NBS (threshold = 2.75) g1>g2")